In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10


In [4]:
# Datasets & DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# image => scale(0, 1) => normalize => (-1 , 1)
transform =transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5 , 0.5))
])

trainset = CIFAR10(root="./Data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./Data", train=False, download=True, transform=transform)

Using downloaded and verified file: ./Data\cifar-10-python.tar.gz
Extracting ./Data\cifar-10-python.tar.gz to ./Data
Files already downloaded and verified


In [5]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./Data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [6]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

# Build the CNN

In [12]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), #kernal = 2 , stride=2

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), #kernal = 2 , stride=2

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2) #kernal = 2 , stride=2
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattiening
        x = self.fc_layers(x)

        return x

In [13]:
model = CNN()


In [14]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Training the CNN

In [17]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()
        
        output = model.forward(images) #FP
        loss = criterion(output, labels) # loss fnx
        loss.backward() # BP
        optimizer.step() # update params

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss=1.359659597239531
epoch=2/10 & loss=0.9300724337320498
epoch=3/10 & loss=0.7472584869169518
epoch=4/10 & loss=0.6208577319941557
epoch=5/10 & loss=0.5165037842052976
epoch=6/10 & loss=0.4151234741672835
epoch=7/10 & loss=0.3339819057708811
epoch=8/10 & loss=0.2627540600231236
epoch=9/10 & loss=0.20033686146940416
epoch=10/10 & loss=0.15616941883864685


In [20]:
# Evaluate our CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100}")    

accuracy = 74.83999999999999


In [22]:
# validation state add hw